In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # disable GPU devices
os.environ["TFDS_DATA_DIR"] = os.path.expanduser("~/tensorflow_datasets")  # default location of tfds database
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
from keras import layers, models, regularizers
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

import tensorflow as tf
import tensorflow_datasets as tfds

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# Turn off logging for TF
import logging
logging.disable(logging.WARNING)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
tf.get_logger().setLevel(logging.ERROR)

from dpmhm.datasets import preprocessing, feature, utils, transformer, query_parameters

In [ ]:
type_dataset = 'meta_dataset' # meta_dataset with DIRG + Paderborn in pre-training and CWRU in final training, images of (64, 64)
# type_dataset = 'meta_dataset_full_bandwidth' # meta_dataset with DIRG + Paderborn in pre-training and CWRU in final training, images of (256, 80)
# type_dataset = 'few_shot_cwru' # Split CWRU in two parts : on for pre-training and one for final training, images of (64, 64)
# type_dataset = 'few_shot_cwru_full_bandwidth' # Split CWRU in two parts : on for pre-training and one for final training, images of (256, 80)
# type_dataset = 'SA_femto' # For Survival analysis on Femto

In [ ]:
outdir = Path('/volatile/home/bm279471/tmp/'+type_dataset)
os.makedirs(outdir, exist_ok=True)

In [ ]:
if type_dataset=='meta_dataset':
    nb_elem_dirg=10000
    nb_elem_paderborn= 10000

    def pipeline(ds_name:str, *, split:str='all', channels:list=[], keys:list=None):
        """Pipeline of preprocessing.

        Parameters
        ----------
        ds_name
            name of the dataset
        split, optional
            split to load, by default 'all'
        channels, optional
            channels to load, by default load all channels simultaneously
        keys, optional
            keys for the ramnification of labels

        """
        ds0 = tfds.load(ds_name, split=split)
        if keys is None:
            keys = query_parameters(ds_name)['keys'].keys()

        compactor = transformer.DatasetCompactor(
            ds0,
            channels=channels, # select all channels simultaneously
            keys=keys,
            # resampling_rate=12000,
            # split_channel=True,  # split multidimensional signals into 1d signals, incompatible with the pretrained VGGish model
        )

        _func = lambda x, sr: feature.spectral_features(
            x, sr, 'spectrogram',
            # n_mfcc=256,
            time_window=0.025, hop_step=0.0125,
            # n_fft=512,
            normalize=False, to_db=True)[0]

        extractor = transformer.FeatureExtractor(compactor.dataset, _func)

        window = transformer.WindowSlider(extractor.dataset, window_size=(64, 64), hop_size=(32, 32))

        labels = list(compactor.full_label_dict.keys())  # need the whole list of labels

        return window.dataset, labels

    ds_list = ['CWRU', 'DIRG', 'Paderborn']
    ds_all = {}

    ds_all['CWRU'] = pipeline('CWRU')
    ds_all['DIRG'] = pipeline('DIRG', split='variation', channels=['A1'])
    ds_all['Paderborn'] = pipeline('Paderborn', split='healthy[:25%]+artificial[:25%]', channels=['vibration', 'current'])

    ds1, ds2, ds3 = ds_all['CWRU'][0], ds_all['DIRG'][0], ds_all['Paderborn'][0]
    lb1, lb2, lb3 = ds_all['CWRU'][1], ds_all['DIRG'][1], ds_all['Paderborn'][1]

    ds2=ds2.shuffle(20000).take(nb_elem_dirg)
    ds3=ds3.shuffle(20000).take(nb_elem_paderborn)
    ds_train = ds2.concatenate(ds3)

    preproc_train = preprocessing.get_mapping_supervised(lb2+lb3)
    preproc_test = preprocessing.get_mapping_supervised(lb1)

    ds_train = utils.restore_shape(
        ds_train.map(preproc_train, num_parallel_calls=tf.data.AUTOTUNE))
    ds_test = utils.restore_shape(
        ds1.map(preproc_test, num_parallel_calls=tf.data.AUTOTUNE))

    eles = list(ds_train.take(10).as_numpy_iterator())
    input_shape = eles[0][0].shape

    ds_train = ds_train.map(lambda x,y: (tf.ensure_shape(x, input_shape), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds_test = ds_test.map(lambda x,y: (tf.ensure_shape(x, input_shape), y), num_parallel_calls=tf.data.AUTOTUNE)

    ds_train=ds_train.map(lambda x, y: (layers.Rescaling(2/(tf.reduce_max(x)-tf.reduce_min(x)), offset=1-tf.reduce_max(x)*2/(tf.reduce_max(x)-tf.reduce_min(x)))(x), y))
    split_train = {'train':0.8, 'val':0.2}
    ds_split_train = utils.split_dataset(ds_train, split_train)

    ds_test=ds_test.map(lambda x, y: (layers.Rescaling(2/(tf.reduce_max(x)-tf.reduce_min(x)), offset=1-tf.reduce_max(x)*2/(tf.reduce_max(x)-tf.reduce_min(x)))(x), y))
    split_test = {'fine-tuning':0.1, 'val':0.1, 'test':0.8}
    ds_split_test = utils.split_dataset(ds_test, split_test, labels=[i+1 for i in range(len(lb1))])

    import json

    ds_split_train['train'].save(str(outdir/'ds_train'))
    ds_split_train['val'].save(str(outdir/'ds_val'))
    ds_split_test['fine-tuning'].save(str(outdir/'ds_train_ft'))
    ds_split_test['val'].save(str(outdir/'ds_val_ft'))
    ds_split_test['test'].save(str(outdir/'ds_test_ft'))

    with open(outdir/'lb1.json', 'w') as fp:
        json.dump(lb1,fp)
    with open(outdir/'lb2.json', 'w') as fp:
        json.dump(lb2,fp)
    with open(outdir/'lb3.json', 'w') as fp:
        json.dump(lb3,fp)

In [ ]:
if type_dataset=='meta_dataset_full_bandwidth':
    nb_elem_dirg=10000
    nb_elem_paderborn= 10000

    def pipeline(ds_name:str, *, split:str='all', channels:list=[], keys:list=None):
        """Pipeline of preprocessing.

        Parameters
        ----------
        ds_name
            name of the dataset
        split, optional
            split to load, by default 'all'
        channels, optional
            channels to load, by default load all channels simultaneously
        keys, optional
            keys for the ramnification of labels

        """
        ds0 = tfds.load(ds_name, split=split)
        if keys is None:
            keys = query_parameters(ds_name)['keys'].keys()

        compactor = transformer.DatasetCompactor(
            ds0,
            channels=channels, # select all channels simultaneously
            keys=keys,
            # resampling_rate=12000,
            # split_channel=True,  # split multidimensional signals into 1d signals, incompatible with the pretrained VGGish model
        )

        _func = lambda x, sr: feature.spectral_features(
            x, sr, 'spectrogram',
            # n_mfcc=256,
            time_window=0.025, hop_step=0.0125,
            # n_fft=512,
            normalize=False, to_db=True)[0]

        extractor = transformer.FeatureExtractor(compactor.dataset, _func)

        window = transformer.WindowSlider(extractor.dataset, window_size=(256, 80), hop_size=40)

        labels = list(compactor.full_label_dict.keys())  # need the whole list of labels

        return window.dataset, labels

    ds_list = ['CWRU', 'DIRG', 'Paderborn']
    ds_all = {}

    ds_all['CWRU'] = pipeline('CWRU')
    ds_all['DIRG'] = pipeline('DIRG', split='variation', channels=['A1'])
    ds_all['Paderborn'] = pipeline('Paderborn', split='healthy[:25%]+artificial[:25%]', channels=['vibration', 'current'])

    ds1, ds2, ds3 = ds_all['CWRU'][0], ds_all['DIRG'][0], ds_all['Paderborn'][0]
    lb1, lb2, lb3 = ds_all['CWRU'][1], ds_all['DIRG'][1], ds_all['Paderborn'][1]

    ds2=ds2.shuffle(20000).take(nb_elem_dirg)
    ds3=ds3.shuffle(20000).take(nb_elem_paderborn)
    ds_train = ds2.concatenate(ds3)

    preproc_train = preprocessing.get_mapping_supervised(lb2+lb3)
    preproc_test = preprocessing.get_mapping_supervised(lb1)

    ds_train = utils.restore_shape(
        ds_train.map(preproc_train, num_parallel_calls=tf.data.AUTOTUNE))
    ds_test = utils.restore_shape(
        ds1.map(preproc_test, num_parallel_calls=tf.data.AUTOTUNE))

    eles = list(ds_train.take(10).as_numpy_iterator())
    input_shape = eles[0][0].shape

    ds_train = ds_train.map(lambda x,y: (tf.ensure_shape(x, input_shape), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds_test = ds_test.map(lambda x,y: (tf.ensure_shape(x, input_shape), y), num_parallel_calls=tf.data.AUTOTUNE)

    ds_train=ds_train.map(lambda x, y: (layers.Rescaling(2/(tf.reduce_max(x)-tf.reduce_min(x)), offset=1-tf.reduce_max(x)*2/(tf.reduce_max(x)-tf.reduce_min(x)))(x), y))
    split_train = {'train':0.8, 'val':0.2}
    ds_split_train = utils.split_dataset(ds_train, split_train)

    ds_test=ds_test.map(lambda x, y: (layers.Rescaling(2/(tf.reduce_max(x)-tf.reduce_min(x)), offset=1-tf.reduce_max(x)*2/(tf.reduce_max(x)-tf.reduce_min(x)))(x), y))
    split_test = {'fine-tuning':0.1, 'val':0.1, 'test':0.8}
    ds_split_test = utils.split_dataset(ds_test, split_test, labels=[i+1 for i in range(len(lb1))])

    import json

    ds_split_train['train'].save(str(outdir/'ds_train'))
    ds_split_train['val'].save(str(outdir/'ds_val'))
    ds_split_test['fine-tuning'].save(str(outdir/'ds_train_ft'))
    ds_split_test['val'].save(str(outdir/'ds_val_ft'))
    ds_split_test['test'].save(str(outdir/'ds_test_ft'))

    with open(outdir/'lb1.json', 'w') as fp:
        json.dump(lb1,fp)
    with open(outdir/'lb2.json', 'w') as fp:
        json.dump(lb2,fp)
    with open(outdir/'lb3.json', 'w') as fp:
        json.dump(lb3,fp)

In [ ]:
if type_dataset=='few_shot_cwru':
    ds0, ds_info = tfds.load(
        'CWRU',
        split='all',
        with_info=True,
    )
    compactor = transformer.DatasetCompactor(ds0,
                                             channels=['DE', 'FE', 'BA'],
                                             keys=['FaultLocation', 'FaultComponent', 'FaultSize'],
                                             resampling_rate=12000)

    # Feature extractor
    # Spectrogram is computed on a time window of 0.025 second every 0.0125 second, then converted to decibel scale.
    _func = lambda x, sr: feature.spectral_features(x, sr, 'spectrogram',
    #                                                 n_mfcc=256,
                                                    time_window=0.025, hop_step=0.0125, n_fft=512,
                                                    normalize=False, to_db=True)[0]

    extractor = transformer.FeatureExtractor(compactor.dataset, _func)

    window = transformer.WindowSlider(extractor.dataset, window_size=(256, 80), hop_size=40) 

    labels = list(compactor.full_label_dict.keys())

    preproc = preprocessing.get_mapping_supervised(labels)
        
    ds_window = window.dataset.map(preproc, num_parallel_calls=tf.data.AUTOTUNE)

    eles = list(ds_window.take(10).as_numpy_iterator())
    input_shape = eles[0][0].shape

    ds_window = ds_window.map(lambda x,y: (tf.ensure_shape(x, input_shape), y), num_parallel_calls=tf.data.AUTOTUNE)

    splits = {'train':0.7, 'transfer':0.1,'val':0.1, 'test':0.1}
    ds_split = utils.split_dataset(ds_window, splits, labels=[i+1 for i in range(len(labels))])

    ds_split['train'].save(str(outdir/'ds_train'))
    ds_split['val'].save(str(outdir/'ds_val'))
    ds_split['transfer'].save(str(outdir/'ds_transfer'))
    ds_split['test'].save(str(outdir/'ds_test'))

    with open(outdir/'lb.json', 'w') as fp:
        json.dump(labels,fp)

In [ ]:
if type_dataset=='few_shot_cwru_full_bandwidth':
    ds0, ds_info = tfds.load(
        'CWRU',
        split='all',
        with_info=True,
    )

    compactor = transformer.DatasetCompactor(ds0,
                                             channels=['DE', 'FE', 'BA'],
                                             keys=['FaultLocation', 'FaultComponent', 'FaultSize'],
                                             resampling_rate=12000)

    # Feature extractor
    # Spectrogram is computed on a time window of 0.025 second every 0.0125 second, then converted to decibel scale.
    _func = lambda x, sr: feature.spectral_features(x, sr, 'spectrogram',
    #                                                 n_mfcc=256,
                                                    time_window=0.025, hop_step=0.0125, n_fft=512,
                                                    normalize=False, to_db=True)[0]

    extractor = transformer.FeatureExtractor(compactor.dataset, _func)

    window = transformer.WindowSlider(extractor.dataset, window_size=(256, 80), hop_size=40) 

    labels = list(compactor.full_label_dict.keys())

    preproc = preprocessing.get_mapping_supervised(labels)
        
    ds_window = window.dataset.map(preproc, num_parallel_calls=tf.data.AUTOTUNE)

    eles = list(ds_window.take(10).as_numpy_iterator())
    input_shape = eles[0][0].shape

    ds_window = ds_window.map(lambda x,y: (tf.ensure_shape(x, input_shape), y), num_parallel_calls=tf.data.AUTOTUNE)

    splits = {'train':0.7, 'transfer':0.1,'val':0.1, 'test':0.1}
    ds_split = utils.split_dataset(ds_window, splits, labels=[i+1 for i in range(len(labels))])

    ds_split['train'].save(str(outdir/'ds_train'))
    ds_split['val'].save(str(outdir/'ds_val'))
    ds_split['transfer'].save(str(outdir/'ds_transfer'))
    ds_split['test'].save(str(outdir/'ds_test'))

    with open(outdir/'lb.json', 'w') as fp:
        json.dump(labels,fp)

In [ ]:
if type_dataset=='SA_femto':
    ds0 = tfds.load('FEMTO', split='train')
    keys=['RemainingUsefulLife', 'TimeElapsed']

    compactor_train = transformer.DatasetCompactor(
        ds0,
        channels=['vibration'], 
        keys=keys,
        resampling_rate=25600, 
        window_size=2560,)

    labels_train = list(compactor_train.full_label_dict.keys())

    _func = lambda x, sr: feature.spectral_features(
        x, sr, 'spectrogram',
        time_window=0.025, hop_step=0.0125,
        normalize=False, to_db=True)[0]

    extractor_train = transformer.FeatureExtractor(compactor_train.dataset, _func)

    window_train = transformer.WindowSlider(extractor_train.dataset, window_size=(496,8), hop_size=8)

    preproc_train = preprocessing.get_mapping_supervised(labels_train)
        
    ds_window_train = window_train.dataset.map(preproc_train, num_parallel_calls=tf.data.AUTOTUNE)

    eles = list(ds_window_train.take(1).as_numpy_iterator())
    input_shape = eles[0][0].shape

    ds_window_train = ds_window_train.map(lambda x,y: (tf.ensure_shape(x, input_shape), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds_window_train = ds_window_train.map(lambda x,y: (layers.Rescaling(2/(tf.reduce_max(x)-tf.reduce_min(x)), offset=1-tf.reduce_max(x)*2/(tf.reduce_max(x)-tf.reduce_min(x)))(x), y))

    splits = {'train':0.8, 'val':0.2}
    ds_split = utils.split_dataset(ds_window_train, splits)

    ds1 = tfds.load('FEMTO', split='test')

    ds_test={}
    labels_test={}
    equivalence_labels_test={}
    for file in ['Bearing1_3', 'Bearing1_4', 'Bearing1_5', 'Bearing1_6', 'Bearing1_7', 'Bearing2_3', 'Bearing2_4', 'Bearing2_5', 'Bearing2_6', 'Bearing2_7', 'Bearing3_3']:
        compactor_test = transformer.DatasetCompactor(
            ds1,
            channels=['vibration'], 
            keys=keys,
            resampling_rate=25600, 
            window_size=2560,
            filters={'Bearing' : file})

        labels_ = list(compactor_test.full_label_dict.keys())

        extractor_test = transformer.FeatureExtractor(compactor_test.dataset, _func)

        window_test = transformer.WindowSlider(extractor_test.dataset, window_size=(496,8), hop_size=8)

        preproc_test = preprocessing.get_mapping_supervised(labels_)
            
        ds_window_test = window_test.dataset.map(preproc_test, num_parallel_calls=tf.data.AUTOTUNE)


        ds_window_test = ds_window_test.map(lambda x,y: (tf.ensure_shape(x, input_shape), y), num_parallel_calls=tf.data.AUTOTUNE)
        ds_window_test = ds_window_test.map(lambda x,y: (layers.Rescaling(2/(tf.reduce_max(x)-tf.reduce_min(x)), offset=1-tf.reduce_max(x)*2/(tf.reduce_max(x)-tf.reduce_min(x)))(x), y))

        ds_test[file]=ds_window_test
        labels_test[file]=labels_
        equivalence_labels_test[file]=compactor_test.full_label_dict

        import json

        equivalence_labels_train=compactor_train.full_label_dict

        outdir = Path('/volatile/home/bm279471/tmp/SA_femto')
        os.makedirs(outdir, exist_ok=True)

        ds_split['train'].save(str(outdir/'ds_train'))
        ds_split['val'].save(str(outdir/'ds_val'))

        with open(outdir/'labels_train.json', 'w') as fp:
            json.dump(labels_train,fp)
        with open(outdir/'equivalence_labels_train.json', 'w') as fp:
            json.dump(equivalence_labels_train, fp)
        for file in ['Bearing1_3', 'Bearing1_4', 'Bearing1_5', 'Bearing1_6', 'Bearing1_7', 'Bearing2_3', 'Bearing2_4', 'Bearing2_5', 'Bearing2_6', 'Bearing2_7', 'Bearing3_3']:
            ds_test[file].save(str(outdir/'ds_test_')+file)
            labels_dir='labels_test_'+file+'.json'
            with open(outdir/labels_dir, 'w') as fp:
                json.dump(labels_test[file],fp)
            equivalence_labels_dir='equivalence_labels_test_'+file+'.json'
            with open(outdir/equivalence_labels_dir, 'w') as fp:
                json.dump(equivalence_labels_test[file], fp)